# Experiment 01: Blend of mlp_4out_emb_bn and LightGBM

Card: [exp01_blend.md](../../experiments/exp01_blend.md)

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, "../../..")  # the ashrae package lives in the homework folder
from ashrae.config import FOLDS, METERS
from ashrae.data import read_processed, split
from ashrae.features import build_features
from ashrae.metrics import rmsle
from ashrae.modeling import MeterMLP, make_loaders, predict
from ashrae.tracking import Experiment

# Imported after torch: loading LightGBM first crashes the kernel on macOS (both ship an OpenMP runtime).
import lightgbm as lgb

SMOKE = os.environ.get("SMOKE") == "1"
NOTEBOOK = "exp01_blend"
LGBMS = ["lgbm", "lgbm_building"]
if SMOKE:
    EXPERIMENT = "ashrae-ensemble-blend-smoke"
    FOLD_NAMES = list(FOLDS)[:2]
    N_TRAIN, N_TRAIN_BIG, N_VAL = 20_000, 40_000, 5_000
    EPOCHS = 1
    LGBM_PARAMS = {"n_estimators": 10, "learning_rate": 0.1, "num_leaves": 63}
    WEIGHTS = np.round(np.linspace(0, 1, 5), 2)
    FIGURE = None
else:
    EXPERIMENT = "ashrae-ensemble-blend"
    FOLD_NAMES = list(FOLDS)
    N_TRAIN, N_TRAIN_BIG, N_VAL = 2_000_000, 4_000_000, 500_000  # rows sampled per fold
    EPOCHS = 20
    LGBM_PARAMS = {"n_estimators": 500, "learning_rate": 0.1, "num_leaves": 63}
    WEIGHTS = np.round(np.linspace(0, 1, 21), 2)  # weight of the MLP
    FIGURE = "../../figures/exp01_blend.png"

experiment = Experiment(EXPERIMENT, NOTEBOOK)

## Training

In [ ]:
df = read_processed()
n_buildings = int(df.building_id.max()) + 1
BUILDING = pd.CategoricalDtype(range(n_buildings))


def with_building(X, rows):
    return X.assign(building_id=rows.building_id.astype(int).astype(BUILDING).to_numpy())


def run_fold(fold, n_train, models):
    """Validation predictions of `models` for one fold, trained on `n_train` rows."""
    train_df, val_df = split(df, FOLDS[fold], n_train, N_VAL)
    year_median = train_df.year_built.median()
    X_train, X_val = build_features(train_df, year_median), build_features(val_df, year_median)
    y_train = np.log1p(train_df.meter_reading.to_numpy(dtype="float32"))
    preds = {}
    if "mlp_4out_emb_bn" in models:
        train_loader, val_loader, _ = make_loaders(train_df, val_df, X_train, X_val)
        model, _ = experiment.fit(lambda: MeterMLP(X_train.shape[1], n_buildings, batch_norm=True), train_loader,
                                  run_name=f"mlp_4out_emb_bn | {n_train:,} rows | {fold}", epochs=EPOCHS,
                                  tags={"model": "mlp_4out_emb_bn", "fold": fold, "n_train": n_train})
        preds["mlp_4out_emb_bn"] = predict(model, val_loader)
    for name in [m for m in models if m in LGBMS]:
        Xtr, Xv = (with_building(X_train, train_df), with_building(X_val, val_df)) if name == "lgbm_building" \
            else (X_train, X_val)
        # n_jobs=1: multi-threaded LightGBM segfaults once torch has loaded its OpenMP runtime
        model = lgb.LGBMRegressor(**LGBM_PARAMS, random_state=0, verbose=-1, n_jobs=1).fit(Xtr, y_train)
        preds[name] = np.clip(model.predict(Xv), 0, None)
    return {"y": np.log1p(val_df.meter_reading.to_numpy(dtype="float32")), "anomaly": val_df.anomaly.to_numpy(),
            "meter": val_df.meter.to_numpy(), "pred": preds}


oof = {}
for fold in FOLD_NAMES:
    oof[fold] = run_fold(fold, N_TRAIN, ["mlp_4out_emb_bn", *LGBMS])
    print(fold + ": " + ", ".join(f"{m} {rmsle(oof[fold]['y'], p):.3f}" for m, p in oof[fold]["pred"].items()))

## Blend weight

In [ ]:
def blend(data, other, w):
    """Weighted average in log1p space, the space RMSLE is measured in."""
    return w * data["pred"]["mlp_4out_emb_bn"] + (1 - w) * data["pred"][other]


def curve(other, rows="all"):
    """RMSLE per weight (rows) and fold (columns)."""
    def score(data, w):
        keep = ~data["anomaly"] if rows == "clean" else slice(None)
        return rmsle(data["y"][keep], blend(data, other, w)[keep])
    return pd.DataFrame({fold: [score(data, w) for w in WEIGHTS] for fold, data in oof.items()}, index=WEIGHTS)


curves = {other: curve(other) for other in LGBMS}
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (other, c) in zip(axes, curves.items()):
    c.plot(ax=ax, alpha=0.5)
    c.mean(axis=1).plot(ax=ax, color="black", lw=2, label="average")
    ax.set(title=f"mlp_4out_emb_bn + {other}", xlabel="MLP weight w", ylabel="RMSLE, all rows")
    ax.legend(fontsize=8)
plt.tight_layout()
if FIGURE:
    fig.savefig(FIGURE, dpi=150, bbox_inches="tight")

## Scores

In [ ]:
rows = {}
for rows_kind in ["all", "clean"]:
    for model in ["mlp_4out_emb_bn", *LGBMS]:
        rows[(rows_kind, model)] = {
            fold: rmsle(d["y"][~d["anomaly"]] if rows_kind == "clean" else d["y"],
                        d["pred"][model][~d["anomaly"]] if rows_kind == "clean" else d["pred"][model])
            for fold, d in oof.items()}
    for other in LGBMS:
        c = curve(other, rows_kind) if rows_kind == "clean" else curves[other]
        # leave one fold out: the weight is chosen on the other folds, on all rows
        chosen = {fold: float(curves[other].drop(columns=fold).mean(axis=1).idxmin()) for fold in FOLD_NAMES}
        rows[(rows_kind, f"blend with {other}")] = {fold: c.loc[chosen[fold], fold] for fold in FOLD_NAMES}
        if rows_kind == "all":
            print(f"blend with {other}: MLP weight chosen per held-out fold {chosen}")

scores = pd.DataFrame(rows).T[FOLD_NAMES]
scores["average"] = scores.mean(axis=1)
display(scores.round(3).rename_axis(["rows", "model"]))

## Error correlation and meters

In [ ]:
corr = {other: np.mean([np.corrcoef(d["pred"]["mlp_4out_emb_bn"][~d["anomaly"]] - d["y"][~d["anomaly"]],
                                     d["pred"][other][~d["anomaly"]] - d["y"][~d["anomaly"]])[0, 1]
                         for d in oof.values()]) for other in LGBMS}
display(pd.Series(corr, name="residual correlation with mlp_4out_emb_bn").round(3).to_frame())

best_other = min(LGBMS, key=lambda o: scores.loc[("all", f"blend with {o}"), "average"])
w_best = float(curves[best_other].mean(axis=1).idxmin())
per_meter = {}
for m, name in METERS.items():
    per_meter[name] = {}
    for label, get in [("mlp_4out_emb_bn", lambda d: d["pred"]["mlp_4out_emb_bn"]), (best_other, lambda d: d["pred"][best_other]),
                       (f"blend, w = {w_best}", lambda d: blend(d, best_other, w_best))]:
        per_meter[name][label] = np.mean([rmsle(d["y"][d["meter"] == m], get(d)[d["meter"] == m]) for d in oof.values()])
    # best weight for this meter alone, on the fold average
    per_meter[name]["best w for this meter"] = WEIGHTS[np.argmin([np.mean([
        rmsle(d["y"][d["meter"] == m], blend(d, best_other, w)[d["meter"] == m]) for d in oof.values()]) for w in WEIGHTS])]
display(pd.DataFrame(per_meter).T.round(3).rename_axis("fold average, all rows"))

## More training rows

In [ ]:
big = {}
for fold in FOLD_NAMES:
    big[fold] = run_fold(fold, N_TRAIN_BIG, ["mlp_4out_emb_bn", best_other])
    assert np.array_equal(big[fold]["y"], oof[fold]["y"])  # same validation rows as above

size_rows = {}
for label, runs in [(f"{N_TRAIN:,} rows", oof), (f"{N_TRAIN_BIG:,} rows", big)]:
    for model in ["mlp_4out_emb_bn", best_other]:
        size_rows[(label, model)] = {fold: rmsle(d["y"], d["pred"][model]) for fold, d in runs.items()}
    size_rows[(label, f"blend, w = {w_best}")] = {fold: rmsle(d["y"], blend(d, best_other, w_best))
                                                 for fold, d in runs.items()}
size_table = pd.DataFrame(size_rows).T[FOLD_NAMES]
size_table["average"] = size_table.mean(axis=1)
display(size_table.round(3).rename_axis(["training rows", "model"]))